# 05. Context rot 측정

입력 길이와 근거 위치를 바꾸되 질문, 모델, 지시, 평가 기준은 고정해야 길이 효과를 비교할 수 있습니다.

## 실행 환경 준비

실습 파일은 `labs/part2/ch08-context-engineering/notebooks/05-Context-rot-측정.ipynb`입니다. 새 커널에서 시작해 첫 코드 셀로 패키지 버전을 맞춘 뒤, 설명과 코드를 위에서부터 순서대로 실행합니다. `.env`는 현재 작업 폴더의 상위 경로에서도 찾습니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
from dotenv import find_dotenv, load_dotenv
import os

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print(".env와 API 키 확인 완료")

.env와 API 키 확인 완료


### 모델 객체를 만듭니다

`.env` 확인이 끝난 뒤 모델 이름과 출력 제한을 지정합니다. 이 셀은 아직 API에 질문을 보내지 않습니다.

In [3]:
# init_chat_model은 provider:model 형식으로 채팅 모델 객체를 만듭니다.
# model은 이후 셀에서 invoke()를 호출할 때 사용합니다.
from langchain.chat_models import init_chat_model

# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=300,
)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 같은 근거를 짧은 입력과 긴 입력에 배치합니다

긴 입력에는 질문과 무관한 문장을 추가합니다. 실제 실험에서는 길이와 위치별로 여러 번 반복합니다.

In [4]:
# evidence와 question은 두 조건에서 동일하게 유지합니다.
evidence = "[E01 p.2] 정기 휴관일은 매주 월요일입니다."
question = "미르시립도서관의 정기 휴관일과 출처를 알려 주세요."
print("질문:", question)
print("근거:", evidence)

질문: 미르시립도서관의 정기 휴관일과 출처를 알려 주세요.
근거: [E01 p.2] 정기 휴관일은 매주 월요일입니다.


### 한 값만 바꾼 두 입력을 만듭니다

`short`와 `long`은 질문과 근거가 같고, 근거 위치도 맨 앞에 고정합니다. `long`에만 무관한 문장을 추가해 입력 길이 한 조건만 바꿉니다.

In [5]:
# 문자열에 120을 곱하면 같은 무관 문장을 120번 연결합니다.
noise = "회의 준비 항목을 확인했습니다. " * 120
prompts = {
    "short": f"""근거:
{evidence}

질문: {question}""",
    "long": f"""근거:
{evidence}
{noise}

질문: {question}""",
}
for name, text in prompts.items():
    print(name, "문자 수:", len(text))

short 문자 수: 66
long 문자 수: 2227


## 3. 두 조건의 실제 사용량을 측정합니다

두 호출의 입력 토큰과 답을 나란히 출력합니다. 두 번의 호출만으로 일반 결론을 내리지는 않습니다.

In [6]:
# 두 prompt를 같은 system 지시로 호출해 입력 길이만 다른 결과를 만듭니다.
# result는 AIMessage이며 results에 보관한 뒤 실제 입력 토큰과 답변을 출력합니다.
results = {}
for name, prompt in prompts.items():
    messages = [("system", "근거만 사용하고 출처를 표시하세요."), ("user", prompt)]
    result = model.invoke(messages)
    usage = result.usage_metadata
    if usage is None:
        raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
    results[name] = result
    print(name, "input_tokens:", usage["input_tokens"])
    print(result.text)

short input_tokens: 65
미르시립도서관의 **정기 휴관일은 매주 월요일**입니다.  
출처: **[E01 p.2]**


long input_tokens: 1145
미르시립도서관의 정기 휴관일은 **매주 월요일**입니다. [E01 p.2]

출처: **[E01 p.2] 정기 휴관일은 매주 월요일입니다.**


## 4. 최소 루브릭으로 확인합니다

자동 문자열 검사는 빠르지만, 대상 기관과 문장의 모순 여부는 사람이 함께 읽어야 합니다.

In [7]:
# int(True)는 1, int(False)는 0이므로 두 문자열 검사를 점수로 더할 수 있습니다.
# score는 요일과 출처 표기가 각각 있는지만 확인하는 0~2점 값입니다.
for name, result in results.items():
    answer = result.text
    score = int("월요일" in answer) + int("E01 p.2" in answer)
    print(name, "자동 점수:", score, "/ 2")

short 자동 점수: 2 / 2
long 자동 점수: 2 / 2


## 정리

이번 두 호출은 측정 방법을 익히는 최소 예제입니다. 반복 횟수와 실행 순서를 기록해야 재현 가능한 실험이 됩니다.

## 실패 재현: 조건마다 질문을 바꿉니다

입력 길이와 질문 난이도를 함께 바꾸면 어느 변화가 점수 차이를 만들었는지 구분할 수 없습니다.

In [8]:
# questions는 조건별 질문 목록이며, set은 중복 질문을 하나로 모읍니다.
# 서로 다른 질문이 두 개 이상이면 비교 조건이 달라졌으므로 ValueError를 냅니다.
def require_same_question(questions: list[str]) -> None:
    if len(set(questions)) != 1:
        raise ValueError("조건마다 질문이 다릅니다.")

### 두 번째 질문 한 개만 바꿉니다

첫 질문은 휴관일, 두 번째 질문은 대출 기간입니다. 입력 길이와 함께 질문까지 달라진 실패를 관찰합니다.

In [9]:
# 비교할 질문 두 개를 목록으로 전달합니다.
try:
    require_same_question([question, "대출 기간은 얼마인가요?"])
except ValueError as error:
    print("관찰한 실패:", error)
else:
    raise AssertionError("질문 불일치 오류가 발생해야 합니다.")

관찰한 실패: 조건마다 질문이 다릅니다.


## 수정: 모든 조건에 같은 질문을 사용합니다

짧은 입력과 긴 입력 모두 앞에서 정의한 휴관일 질문을 사용합니다.

In [10]:
# fixed_questions는 두 조건 모두에 같은 question 값을 사용합니다.
# 검사 함수를 통과한 뒤 고유 질문 수가 하나인지 다시 확인합니다.
fixed_questions = [question, question]
require_same_question(fixed_questions)
assert len(set(fixed_questions)) == 1
print("수정 후 질문 수:", len(fixed_questions))

수정 후 질문 수: 2
